# 🧠 LRU Cache — From Zero to Hero

> **A complete, beginner-friendly guide to understanding, implementing, and visualizing the Least Recently Used (LRU) Cache.**

---

### What You'll Learn
| # | Topic |
|---|-------|
| 1 | What is a cache and why do we need one? |
| 2 | The LRU eviction policy explained |
| 3 | Choosing the right data structures |
| 4 | Building a Doubly Linked List node |
| 5 | Full LRU Cache implementation |
| 6 | Design choices & assumptions |
| 7 | Edge cases |
| 8 | Unit tests |
| 9 | Mock database setup |
| 10 | Interactive Streamlit UI for cache visualization |

## 📦 Section 1 — Import Required Libraries

In [ ]:
# Section 1: Import Required Libraries

from collections import OrderedDict   # Python's built-in ordered dict
import sqlite3                         # Lightweight in-memory database
import time                            # For simulating database latency
import random                          # For generating sample data
import functools                       # For functools.lru_cache comparison
import pandas as pd                    # For nice table displays

print("✅ All libraries imported successfully!")

## 💡 Section 2 — Core Concepts of Caching and the LRU Policy

### What is a Cache?

Imagine you're studying at a library. You have:
- **Your desk** (small, fast to reach) — this is the **cache**
- **The library shelves** (huge, but slow to walk to) — this is the **database / disk / remote server**

You keep the books you use most often **on your desk**. When your desk is full and you need a new book, you put back the one you haven't touched in the longest time. That's **LRU — Least Recently Used**.

### Why Do We Need Caching?
| Without Cache | With Cache |
|:---|:---|
| Every request goes to the database | Frequently accessed data is served from memory |
| Slow (disk I/O, network round-trip) | Fast (in-memory lookup) |
| High load on the database | Reduced database load |

### Eviction Policies Compared
| Policy | Evicts | Pros | Cons |
|--------|--------|------|------|
| **LRU** (Least Recently Used) | Item not accessed for the longest time | Great for temporal locality | Needs tracking of access order |
| **FIFO** (First In, First Out) | Oldest inserted item | Simple to implement | Doesn't consider access frequency |
| **LFU** (Least Frequently Used) | Item accessed the fewest times | Great for skewed workloads | Complex bookkeeping |
| **Random** | A random item | Simplest | Unpredictable evictions |

In [ ]:
# The Problem: Unbounded Caching
# A plain dictionary grows forever — that's dangerous in production!

unbounded_cache = {}

print("🔴 Problem: A plain dict has NO size limit.\n")
for i in range(10):
    unbounded_cache[f"key_{i}"] = f"value_{i}"
    print(f"  Added key_{i}  →  cache size = {len(unbounded_cache)}")

print(f"\n⚠️  Cache has {len(unbounded_cache)} items and will keep growing forever!")
print("   In a real system with millions of keys, this eats all your memory.")
print("\n✅ Solution: Set a MAXIMUM CAPACITY and EVICT old items → that's what LRU Cache does.")

# ASCII Art: How LRU Works Step-by-Step
lru_demo = """
╔══════════════════════════════════════════════════════════════════════╗
║              LRU Cache (capacity = 3) — Step by Step            ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  Step 1: PUT(A)     →  [A]              (1 item)                ║
║  Step 2: PUT(B)     →  [B, A]           (2 items)               ║
║  Step 3: PUT(C)     →  [C, B, A]        (3 items — FULL!)       ║
║  Step 4: GET(A)     →  [A, C, B]        (A moves to front)      ║
║  Step 5: PUT(D)     →  [D, A, C]        (B evicted — LRU!)      ║
║            ↑                    ↑                                ║
║         newest              oldest                               ║
║                                                                  ║
║  Most Recently Used ◄───────────────────► Least Recently Used     ║
║       (HEAD)                                  (TAIL)             ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(lru_demo)

## 🏗️ Section 3 — Data Structure Choices: HashMap + Doubly Linked List

### The Requirements
We need **two operations** to be **O(1)** (constant time):
1. **`get(key)`** — look up a value by key
2. **`put(key, value)`** — insert or update, and evict the LRU item if full

### Why Not Just a Dictionary?
A Python `dict` gives us O(1) lookup, but it has **no inherent ordering**. We can't efficiently know which key was used least recently.

### Why Not Just a List?
A list preserves insertion order, but:
- **Finding** an item is O(n) (linear scan)
- **Moving** an item to the front is O(n) (shifting elements)

### The Winning Combination 🏆

- **HashMap (dict)**: O(1) lookup by key → gives us the **node** directly
- **Doubly Linked List**: O(1) removal & insertion → lets us **move** a node to the front or **remove** the tail

### Why DOUBLY Linked? (Not Singly Linked?)
With a **singly** linked list, to remove a node you need the **previous** node, which requires O(n) traversal. A **doubly** linked list stores a `prev` pointer, so removal is O(1).

In [ ]:
# Complexity Comparison Table

comparison = pd.DataFrame({
    "Data Structure": [
        "Dictionary only",
        "List only",
        "Singly Linked List + Dict",
        "Doubly Linked List + Dict ✅",
    ],
    "get(key)": ["O(1)", "O(n)", "O(1)", "O(1)"],
    "put(key, value)": ["O(1)", "O(n)", "O(n)*", "O(1)"],
    "Evict LRU": ["O(n)", "O(1)", "O(1)", "O(1)"],
    "Move to Front": ["N/A", "O(n)", "O(n)*", "O(1)"],
    "Space": ["O(n)", "O(n)", "O(n)", "O(n)"],
    "Verdict": ["❌ No ordering", "❌ Slow lookup", "❌ Slow removal", "✅ Perfect!"],
})

print("📊 Time Complexity Comparison of Data Structure Approaches:\n")
display(comparison)
print("\n* Singly linked list needs O(n) to find the previous node for removal.")

## 🔗 Section 4 — Implementing the Doubly Linked List Node

Each node in our doubly linked list stores:
- `key` — so we can remove the entry from the HashMap when evicting
- `value` — the cached data
- `prev` — pointer to the previous node
- `next` — pointer to the next node

In [ ]:
# Section 4: The Node Class

class Node:
    """A node in a doubly linked list used by the LRU Cache."""

    def __init__(self, key=0, value=0):
        self.key = key
        self.value = value
        self.prev = None
        self.next = None

    def __repr__(self):
        return f"Node(key={self.key}, val={self.value})"


# --- Demo: manually linking nodes ---
print("🔗 Let's manually create and link three nodes:\n")

node_a = Node("A", 100)
node_b = Node("B", 200)
node_c = Node("C", 300)

# Link them: A ⟷ B ⟷ C
node_a.next = node_b
node_b.prev = node_a
node_b.next = node_c
node_c.prev = node_b

# Traverse forward
print("  Forward traversal:")
current = node_a
path = []
while current:
    path.append(str(current))
    current = current.next
print("    " + " ⟷ ".join(path))

# Traverse backward
print("\n  Backward traversal:")
current = node_c
path = []
while current:
    path.append(str(current))
    current = current.prev
print("    " + " ⟷ ".join(path))

## ⚙️ Section 5 — Implementing the Full LRU Cache Class

The design uses **sentinel (dummy) head and tail nodes** so we never have to deal with `None` checks at the boundaries. The real data nodes live **between** the sentinels.

```
HEAD ⟷ [Most Recent] ⟷ ... ⟷ [Least Recent] ⟷ TAIL
 (dummy)                                       (dummy)
```

### Key Methods
| Method | Purpose |
|--------|---------|
| `_add_node(node)` | Insert a node right after HEAD (most recent position) |
| `_remove_node(node)` | Unlink a node from wherever it is in the list |
| `_move_to_head(node)` | Remove + re-add at front (marks as most recently used) |
| `_pop_tail()` | Remove the node just before TAIL (the LRU item) |
| `get(key)` | Lookup + move to head; returns -1 on miss |
| `put(key, value)` | Insert/update + evict LRU if over capacity |

In [ ]:
# Section 5: Full LRU Cache Implementation

class LRUCache:
    """
    Least Recently Used (LRU) Cache implementation using
    a HashMap (dict) + Doubly Linked List with sentinel nodes.

    Time Complexity:  O(1) for both get() and put()
    Space Complexity: O(capacity)
    """

    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = {}  # key → Node

        # Sentinel (dummy) nodes — simplify edge-case handling
        self.head = Node(key="HEAD", value="SENTINEL")
        self.tail = Node(key="TAIL", value="SENTINEL")
        self.head.next = self.tail
        self.tail.prev = self.head

        print(f"🆕 Created LRU Cache with capacity = {capacity}")

    # —— Internal helpers ——

    def _add_node(self, node):
        """Add a node right after HEAD (most recently used position)."""
        node.prev = self.head
        node.next = self.head.next
        self.head.next.prev = node
        self.head.next = node

    def _remove_node(self, node):
        """Remove a node from its current position in the list."""
        prev_node = node.prev
        next_node = node.next
        prev_node.next = next_node
        next_node.prev = prev_node

    def _move_to_head(self, node):
        """Move an existing node to the front (most recently used)."""
        self._remove_node(node)
        self._add_node(node)

    def _pop_tail(self):
        """Remove and return the node just before TAIL (the LRU item)."""
        lru_node = self.tail.prev
        self._remove_node(lru_node)
        return lru_node

    # —— Public API ——

    def get(self, key):
        """
        Retrieve value for key. Returns -1 if key is not in cache.
        Moves the accessed node to the head (most recently used).
        """
        if key in self.cache:
            node = self.cache[key]
            self._move_to_head(node)
            print(f"  ✅ GET({key}) → {node.value}  [CACHE HIT — moved to front]")
            return node.value
        else:
            print(f"  ❌ GET({key}) → -1  [CACHE MISS]")
            return -1

    def put(self, key, value):
        """
        Insert or update key-value pair.
        If cache is at capacity, evict the LRU item (tail).
        """
        if key in self.cache:
            node = self.cache[key]
            node.value = value
            self._move_to_head(node)
            print(f"  🔄 PUT({key}, {value})  [UPDATED existing key — moved to front]")
        else:
            new_node = Node(key, value)
            self.cache[key] = new_node
            self._add_node(new_node)
            print(f"  ➕ PUT({key}, {value})  [INSERTED new key]", end="")

            if len(self.cache) > self.capacity:
                evicted = self._pop_tail()
                del self.cache[evicted.key]
                print(f"  → 🗑️ EVICTED ({evicted.key}, {evicted.value})")
            else:
                print()

    def display(self):
        """Return a string showing the current cache state (head → tail)."""
        items = []
        current = self.head.next
        while current != self.tail:
            items.append(f"({current.key}: {current.value})")
            current = current.next
        state = " ⟷ ".join(items) if items else "(empty)"
        return f"  📋 Cache [{len(self.cache)}/{self.capacity}]: HEAD ⟷ {state} ⟷ TAIL"


print("✅ LRU Cache class defined!")

In [ ]:
# Step-by-Step Walkthrough

print("=" * 60)
print("  🚶 WALKTHROUGH: LRU Cache with capacity = 3")
print("=" * 60)

cache = LRUCache(capacity=3)

steps = [
    ("put", 1, "Apple"),
    ("put", 2, "Banana"),
    ("put", 3, "Cherry"),
    ("get", 2, None),       # Access Banana → moves to front
    ("put", 4, "Date"),     # Cache full! Evicts key=1 (Apple — LRU)
    ("get", 1, None),       # Miss — was evicted
    ("get", 3, None),       # Hit — Cherry moves to front
    ("put", 5, "Elderberry"),  # Evicts key=2 (Banana — now LRU)
]

for i, (op, key, val) in enumerate(steps, 1):
    print(f"\n── Step {i}: {op.upper()}({key}{', ' + repr(val) if val else ''}) ──")
    if op == "put":
        cache.put(key, val)
    else:
        cache.get(key)
    print(cache.display())

## 📐 Section 6 — Design Choices and Assumptions

| # | Design Choice | Rationale |
|---|--------------|----------|
| 1 | **Fixed capacity** set at initialization | Keeps memory usage predictable |
| 2 | **No TTL (time-to-live)** | Simplicity — items only leave via eviction |
| 3 | **Not thread-safe** | Single-threaded assumption; add `threading.Lock` for concurrency |
| 4 | **Keys must be hashable** | Required by the underlying `dict` |
| 5 | **Returns -1 on cache miss** | Follows LeetCode convention; could also return `None` or raise `KeyError` |
| 6 | **Sentinel nodes** for head/tail | Eliminates `None` boundary checks in every operation |
| 7 | **Values can be any type** | Flexible — strings, ints, dicts, objects |

In [ ]:
# Demonstrating Design Assumptions

# 1. Capacity of 0 — nothing can ever be stored
print("─── Capacity = 0 ───")
c0 = LRUCache(0)
c0.put("x", 10)
print(c0.display())
result = c0.get("x")
print(f"  Result: {result}")
print()

# 2. Unhashable keys — dict will raise TypeError
print("─── Unhashable key (list) ───")
try:
    c_test = LRUCache(3)
    c_test.put([1, 2], "bad key")  # Lists are not hashable!
except TypeError as e:
    print(f"  ❌ TypeError: {e}")
    print("  → Keys must be hashable (int, str, tuple, etc.)")
print()

# 3. Any value type works
print("─── Values can be anything ───")
c_val = LRUCache(3)
c_val.put("dict_val", {"nested": True, "count": 42})
c_val.put("list_val", [1, 2, 3])
c_val.put("str_val", "hello")
print(c_val.display())

## 🧪 Section 7 — Handling Edge Cases

Let's systematically test tricky situations that could break a naive implementation.

In [ ]:
# Edge Case Tests

print("=" * 60)
print("  🧪 EDGE CASE TESTING")
print("=" * 60)

# ── Edge Case 1: Capacity = 1 ──
print("\n── Edge Case 1: Cache with capacity = 1 ──")
c1 = LRUCache(1)
c1.put("A", 1)
print(c1.display())
c1.put("B", 2)  # Should evict A
print(c1.display())
c1.get("A")     # Should miss
print(c1.display())

# ── Edge Case 2: Get on missing key ──
print("\n── Edge Case 2: Get on a key that doesn't exist ──")
c2 = LRUCache(3)
c2.get("nonexistent")

# ── Edge Case 3: Update existing key ──
print("\n── Edge Case 3: Update an existing key's value ──")
c3 = LRUCache(3)
c3.put("X", 100)
c3.put("Y", 200)
c3.put("Z", 300)
print(c3.display())
c3.put("X", 999)  # Update X — should move to front
print(c3.display())

# ── Edge Case 4: Multiple accesses to same key ──
print("\n── Edge Case 4: Repeated access to the same key ──")
c4 = LRUCache(3)
c4.put(1, "A")
c4.put(2, "B")
c4.put(3, "C")
for _ in range(3):
    c4.get(1)  # Each access moves key 1 to front
print(c4.display())

# ── Edge Case 5: Fill, evict all, refill ──
print("\n── Edge Case 5: Full eviction cycle ──")
c5 = LRUCache(2)
c5.put("a", 1)
c5.put("b", 2)
print(c5.display())
c5.put("c", 3)  # Evicts "a"
c5.put("d", 4)  # Evicts "b"
print(c5.display())
c5.get("a")     # Miss
c5.get("b")     # Miss

## ✅ Section 8 — Unit Testing the LRU Cache

In [ ]:
# Section 8: Unit Tests

import unittest
import io, sys

class TestLRUCache(unittest.TestCase):
    """Comprehensive tests for the LRU Cache implementation."""

    def setUp(self):
        self._stdout = sys.stdout
        sys.stdout = io.StringIO()

    def tearDown(self):
        sys.stdout = self._stdout

    def test_put_and_get(self):
        cache = LRUCache(2)
        cache.put(1, 10)
        cache.put(2, 20)
        self.assertEqual(cache.get(1), 10)
        self.assertEqual(cache.get(2), 20)

    def test_get_missing_key(self):
        cache = LRUCache(2)
        self.assertEqual(cache.get(99), -1)

    def test_eviction_removes_lru(self):
        cache = LRUCache(2)
        cache.put(1, 10)
        cache.put(2, 20)
        cache.put(3, 30)   # Evicts key=1
        self.assertEqual(cache.get(1), -1)
        self.assertEqual(cache.get(2), 20)
        self.assertEqual(cache.get(3), 30)

    def test_access_prevents_eviction(self):
        cache = LRUCache(2)
        cache.put(1, 10)
        cache.put(2, 20)
        cache.get(1)        # Makes key=1 most recent
        cache.put(3, 30)    # Evicts key=2 (not key=1)
        self.assertEqual(cache.get(1), 10)
        self.assertEqual(cache.get(2), -1)

    def test_update_existing_key(self):
        cache = LRUCache(2)
        cache.put(1, 10)
        cache.put(2, 20)
        cache.put(1, 100)   # Update key=1
        self.assertEqual(cache.get(1), 100)
        cache.put(3, 30)    # Evicts key=2
        self.assertEqual(cache.get(2), -1)

    def test_capacity_never_exceeded(self):
        cache = LRUCache(3)
        for i in range(100):
            cache.put(i, i * 10)
            self.assertLessEqual(len(cache.cache), 3)

    def test_capacity_one(self):
        cache = LRUCache(1)
        cache.put(1, 10)
        self.assertEqual(cache.get(1), 10)
        cache.put(2, 20)
        self.assertEqual(cache.get(1), -1)
        self.assertEqual(cache.get(2), 20)

    def test_repeated_get(self):
        cache = LRUCache(2)
        cache.put(1, 10)
        cache.put(2, 20)
        for _ in range(10):
            self.assertEqual(cache.get(1), 10)
        self.assertEqual(cache.get(1), 10)

    def test_string_keys(self):
        cache = LRUCache(2)
        cache.put("hello", "world")
        self.assertEqual(cache.get("hello"), "world")


# Run the tests
suite = unittest.TestLoader().loadTestsFromTestCase(TestLRUCache)
runner = unittest.TextTestRunner(verbosity=2, stream=sys.stderr)
result = runner.run(suite)

if result.wasSuccessful():
    print(f"\n🎉 All {result.testsRun} tests passed!")
else:
    print(f"\n⚠️ {len(result.failures)} failure(s), {len(result.errors)} error(s)")

## 🗄️ Section 9 — Setting Up a Mock Database

We'll create an in-memory SQLite database with a **products** table and simulate real-world database latency with `time.sleep()`.

In [ ]:
# Section 9: Mock Database Setup

def create_database():
    """Create an in-memory SQLite database with sample product data."""
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE products (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            price REAL NOT NULL,
            category TEXT NOT NULL,
            description TEXT
        )
    """)

    products = [
        (1,  "Wireless Mouse",       29.99,  "Electronics",  "Ergonomic wireless mouse with USB receiver"),
        (2,  "Mechanical Keyboard",   79.99,  "Electronics",  "RGB backlit mechanical keyboard, blue switches"),
        (3,  "USB-C Hub",             45.00,  "Electronics",  "7-in-1 USB-C hub with HDMI and ethernet"),
        (4,  "Monitor Stand",         34.50,  "Furniture",    "Adjustable aluminum monitor stand"),
        (5,  "Desk Lamp",             22.99,  "Furniture",    "LED desk lamp with 3 brightness levels"),
        (6,  "Webcam HD",             59.99,  "Electronics",  "1080p webcam with built-in microphone"),
        (7,  "Notebook A5",            5.99,  "Stationery",   "Ruled A5 notebook, 200 pages"),
        (8,  "Gel Pen Set",            8.49,  "Stationery",   "Pack of 12 assorted gel pens"),
        (9,  "Laptop Sleeve",         19.99,  "Accessories",  "15-inch neoprene laptop sleeve"),
        (10, "Mouse Pad XL",          14.99,  "Accessories",  "Extended gaming mouse pad, 900x400mm"),
        (11, "Headphones",            49.99,  "Electronics",  "Over-ear noise-cancelling headphones"),
        (12, "Phone Charger",         12.99,  "Electronics",  "Fast-charge USB-C cable, 2m"),
        (13, "Standing Desk Mat",     39.99,  "Furniture",    "Anti-fatigue standing desk mat"),
        (14, "Cable Organizer",        9.99,  "Accessories",  "Silicone cable management clips, 10-pack"),
        (15, "Bluetooth Speaker",     35.00,  "Electronics",  "Portable Bluetooth 5.0 speaker"),
        (16, "Sticky Notes",           3.99,  "Stationery",   "Neon sticky notes, 6 pads"),
        (17, "Whiteboard Markers",     6.49,  "Stationery",   "Dry-erase markers, 8 colors"),
        (18, "Ergonomic Chair",      299.99,  "Furniture",    "Mesh-back ergonomic office chair"),
        (19, "Power Strip",           18.99,  "Electronics",  "6-outlet surge protector with USB ports"),
        (20, "Screen Cleaner Kit",     7.99,  "Accessories",  "Microfiber cloth + cleaning spray"),
        (21, "External SSD 1TB",      89.99,  "Electronics",  "Portable NVMe SSD, USB 3.2"),
        (22, "Wrist Rest",            11.99,  "Accessories",  "Memory foam keyboard wrist rest"),
        (23, "Desk Organizer",        24.99,  "Furniture",    "Bamboo desktop organizer with drawers"),
        (24, "Highlighters",           4.99,  "Stationery",   "Pastel highlighters, 6-pack"),
        (25, "Webcam Light",          15.99,  "Electronics",  "Ring light clip for video calls"),
    ]

    cursor.executemany("INSERT INTO products VALUES (?, ?, ?, ?, ?)", products)
    conn.commit()
    print(f"✅ Database created with {len(products)} products!\n")
    return conn


db = create_database()

df = pd.read_sql_query("SELECT * FROM products", db)
display(df)

In [ ]:
# Database fetch function with simulated latency

DB_LATENCY = 0.3  # seconds — simulates a slow database call

def fetch_from_db(conn, product_id):
    """Fetch a product from the database by ID. Includes simulated latency."""
    time.sleep(DB_LATENCY)
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM products WHERE id = ?", (product_id,))
    row = cursor.fetchone()
    if row:
        return dict(row)
    return None


# Demo: time a raw database call
print("⏱️  Timing a raw database fetch (no cache):\n")
for pid in [1, 5, 99]:
    start = time.time()
    result = fetch_from_db(db, pid)
    elapsed = time.time() - start
    status = result["name"] if result else "NOT FOUND"
    print(f"  Product ID {pid:>2}: {status:<25}  ({elapsed*1000:.0f} ms)")

In [ ]:
# Cached vs Uncached comparison

class CachedDatabase:
    """Wraps database access with an LRU Cache layer."""

    def __init__(self, db_conn, cache_capacity=5):
        self.db = db_conn
        self.cache = LRUCache(cache_capacity)
        self.hits = 0
        self.misses = 0
        self.access_log = []

    def query(self, product_id):
        """Query with cache-first strategy."""
        start = time.time()

        cached = self.cache.cache.get(product_id)
        if cached:
            self.cache._move_to_head(cached)
            result = cached.value
            elapsed_ms = (time.time() - start) * 1000
            self.hits += 1
            self.access_log.append((time.time(), product_id, "HIT", elapsed_ms))
            return result

        result = fetch_from_db(self.db, product_id)
        elapsed_ms = (time.time() - start) * 1000
        self.misses += 1
        self.access_log.append((time.time(), product_id, "MISS", elapsed_ms))

        if result is not None:
            old_stdout = sys.stdout
            sys.stdout = io.StringIO()
            self.cache.put(product_id, result)
            sys.stdout = old_stdout

        return result

    def get_stats(self):
        total = self.hits + self.misses
        hit_rate = (self.hits / total * 100) if total > 0 else 0
        return {
            "total_requests": total,
            "cache_hits": self.hits,
            "cache_misses": self.misses,
            "hit_rate": f"{hit_rate:.1f}%",
        }

    def get_cache_contents(self):
        items = []
        current = self.cache.head.next
        pos = 1
        while current != self.cache.tail:
            items.append({"Position": pos, "Product ID": current.key, **current.value})
            current = current.next
            pos += 1
        return items


# —— Demo: Compare cached vs uncached ——
print("=" * 60)
print("  🏎️  CACHED vs 🐢 UNCACHED — Performance Comparison")
print("=" * 60)

cached_db = CachedDatabase(db, cache_capacity=5)

access_pattern = [1, 2, 3, 1, 4, 2, 5, 1, 6, 3, 1, 2]

print(f"\nAccess pattern: {access_pattern}\n")

for pid in access_pattern:
    start = time.time()
    result = cached_db.query(pid)
    elapsed = (time.time() - start) * 1000
    name = result["name"] if result else "NOT FOUND"
    hit_miss = cached_db.access_log[-1][2]
    icon = "✅" if hit_miss == "HIT" else "🔍"
    print(f"  {icon} Product {pid:>2} → {name:<25} [{hit_miss}]  {elapsed:>6.1f} ms")

print(f"\n📊 Stats: {cached_db.get_stats()}")
print(f"\n📋 Current cache contents:")
display(pd.DataFrame(cached_db.get_cache_contents()))

## 🖥️ Section 10 — Bonus: Python's Built-in LRU Cache

Python includes `functools.lru_cache` and `collections.OrderedDict` that achieve similar results. Let's see how they compare to our hand-rolled version.

In [ ]:
# Built-in Approach 1: functools.lru_cache

@functools.lru_cache(maxsize=3)
def cached_square(n):
    """Compute square — cached automatically."""
    print(f"    ⚙️  Computing {n}² (cache miss)")
    return n * n

print("── functools.lru_cache (maxsize=3) ──\n")
for x in [2, 3, 4, 2, 5, 3]:
    result = cached_square(x)
    print(f"  cached_square({x}) = {result}")
    info = cached_square.cache_info()
    print(f"    Stats: hits={info.hits}, misses={info.misses}, size={info.currsize}\n")

# Built-in Approach 2: OrderedDict-based LRU

class LRUCacheOrderedDict:
    """LRU Cache using Python's OrderedDict (simpler but less educational)."""

    def __init__(self, capacity):
        self.capacity = capacity
        self.cache = OrderedDict()

    def get(self, key):
        if key in self.cache:
            self.cache.move_to_end(key)
            return self.cache[key]
        return -1

    def put(self, key, value):
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            self.cache.popitem(last=False)

print("\n── OrderedDict-based LRU Cache ──\n")
od_cache = LRUCacheOrderedDict(3)
for k, v in [(1, "A"), (2, "B"), (3, "C")]:
    od_cache.put(k, v)
print(f"  After inserting 1,2,3: {list(od_cache.cache.items())}")

od_cache.get(1)
print(f"  After GET(1):          {list(od_cache.cache.items())}")

od_cache.put(4, "D")
print(f"  After PUT(4,'D'):      {list(od_cache.cache.items())}  ← key=2 evicted!")

print("\n📝 Note: OrderedDict is convenient but understanding the DLL+HashMap")
print("   approach is essential for interviews and systems design.")

## 🌐 Section 11 — Streamlit UI: Interactive Cache Visualization

The Streamlit app is in a **separate file** (`streamlit_app.py`) because Streamlit runs as its own server.

### How to Run
```bash
pip install streamlit pandas
streamlit run streamlit_app.py
```

The app provides:
- **Sidebar controls**: cache capacity slider, clear cache button, bulk random access, product catalog
- **Lookup panel**: search by product ID with HIT/MISS visual feedback
- **Stats dashboard**: total requests, hit rate, average times, bar charts
- **Cache contents table**: ordered from MRU to LRU with doubly linked list visualization
- **Color-coded access log**: green for hits, red for misses

## 📊 Section 12 — Complexity Analysis Summary

| Operation | Our LRU Cache | OrderedDict LRU | Plain Dict (no eviction) |
|-----------|:---:|:---:|:---:|
| `get(key)` | **O(1)** | O(1) | O(1) |
| `put(key, value)` | **O(1)** | O(1) | O(1) |
| `evict LRU` | **O(1)** | O(1) | O(n) — find min |
| Space | O(capacity) | O(capacity) | O(n) — unbounded |

### Why is everything O(1)?
- **HashMap** → O(1) key lookup to find the node
- **Doubly Linked List** → O(1) to remove any node (we have `prev` pointer) and O(1) to insert at head
- **Eviction** → O(1) because the LRU item is always right before the tail sentinel

### When to Use LRU Cache
- Database query results
- API response caching
- DNS lookups
- Web page / image caching
- Any scenario with **temporal locality** (recently accessed items are likely to be accessed again soon)

---

**Next step:** Run `streamlit run streamlit_app.py` to launch the interactive visualization!